# Week 4 · AdaBoost and gradient boosting, one small piece at a time

Your log on Oct 30: *"i need you to divide the part where we build the adaboost into smaller chunks and recap this in next week's session. same goes for the Gradient boosting."* Here they are. Each algorithm is cut into pieces of 1–3 lines. Each piece gets one picture, one hand calculation, and one exercise whose **Example** shows the exact input and output. The last piece of each half just calls the earlier ones, so by then the whole algorithm is something you've already written. Same small datasets as Week 3, so you can check every number against what you've seen.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **stump**: A tree with one question. In 1-D: "predict $s$ if $x > t$, otherwise $-s$" (classification), or "predict $v_L$ if $x \le t$, otherwise $v_R$" (regression).
- **weight $w_i$**: How much point $i$ counts in AdaBoost. All weights add up to 1.
- **residual $r_i$**: $y_i - F(x_i)$: what the model so far still gets wrong.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "01")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** Both algorithms are a loop that adds one stump per round. Top: AdaBoost re-weights the **points**. Bottom: gradient boosting re-targets the **labels** (residuals).

In [ ]:
fig_boost_map()

| Section | Piece | Exercise | What it does |
|---|---|---|---|
| 1 | the whole AdaBoost loop |  | what we're building, on the 8 points from Week 3 |
| 2 | a stump's predictions | 1 | turns a threshold and a sign into $\pm 1$ for every point |
| 3 | weighted error | 2 | how wrong the stump is, counting heavy points more |
| 4 | the best stump | 3 | tries every threshold and sign, keeps the smallest weighted error |
| 5 | its vote $\alpha$ | 4 | accurate stumps get a bigger say |
| 6 | reweighting | 5 | the points it got wrong count more next round |
| 7 | the loop, and predicting | 6–7 | pieces 2–6 in a loop of 5 lines |
| 8 | the whole gradient boosting loop |  | on the 4 points from Week 3 |
| 9 | residuals | 8 | what's still wrong: $y - F$ |
| 10 | a stump for residuals | 9–10 | the split that best predicts the residuals |
| 11 | the update, and predicting | 11–12 | add a fraction of the stump, repeat |
| 12 | side by side |  | what the two have in common |

## 0. Recap first · 10 min, closed book

**Quick check:** A regression leaf holds $3, 5, 10$. What does it predict, and what is its SSE?

<details><summary>Answer</summary>

The mean, 6. SSE $= 9 + 1 + 16 = 26$. [Week 3 · 04 §5 ↗](../week3/04_decision_trees.ipynb)

</details>

**Quick check:** Bagging vs boosting: which reduces variance, which reduces bias, and which kind of tree does each use?

<details><summary>Answer</summary>

Bagging averages **deep** trees (low bias, high variance) to cut variance. Boosting adds **small** trees (high bias, low variance) one after another to cut bias. [Week 3 · 06 §4 ↗](../week3/06_ensembles.ipynb)

</details>

**Quick check:** `np.where(np.array([1, 4, 6]) > 3, 1, -1)` is ...?

<details><summary>Answer</summary>

`[-1  1  1]`: 1 where the condition is True, $-1$ where it's False.

</details>

**Quick check:** `w = np.array([0.1, 0.2, 0.3, 0.4])`, `wrong = np.array([True, False, True, False])`. What is `w[wrong].sum()`?

<details><summary>Answer</summary>

$0.1 + 0.3 = 0.4$: a True/False mask keeps the True positions. [Week 1 · NumPy §6 ↗](../week1/02_numpy_lesson.ipynb)

</details>

**Quick check:** Why is the residual $y - F$ the **negative gradient** of the squared loss?

<details><summary>Answer</summary>

$L = \frac{1}{2}(y - F)^2$, so $\frac{\partial L}{\partial F} = -(y - F)$. The negative of that is $y - F$. [Week 3 · 06 §6 ↗](../week3/06_ensembles.ipynb)

</details>

## 1. AdaBoost: the whole loop first

> ↩ **Recap** · [Week 3 · 06 §5 (AdaBoost by hand) ↗](../week3/06_ensembles.ipynb)
>
> 8 points on a line, $x = 1, \dots, 8$, labels $+ + - - + + - -$. Three rounds gave stumps at 2.5, 6.5, 4.5 with votes 0.549, 0.805, 0.693, and their weighted vote got all 8 right.

In [ ]:
x = np.arange(1, 9.0)                          # [1. 2. ... 8.]   shape (8,)
y = np.array([1, 1, -1, -1, 1, 1, -1, -1.0])   # labels +1 / -1   shape (8,)

🖼 **Picture.** Three rounds. Dot size is a point's weight going into the round; ✗ marks the stump's mistakes. Notice how the mistakes of one round get bigger in the next.

In [ ]:
fig_adaboost_rounds()

**The whole algorithm is five lines.** Every line is one piece of this notebook:

| Line | In words | Section |
|---|---|---|
| `w = np.full(n, 1 / n)` | every point starts with the same weight | (given) |
| `t, s, err = best_stump(x, y, w)` | the stump with the smallest **weighted** error | 2, 3, 4 |
| `alpha = stump_alpha(err)` | how much its vote counts | 5 |
| `w = update_weights(w, y, h, alpha)` | the points it got wrong gain weight | 6 |
| `model.append((t, s, alpha))`, repeat | remember the stump, go again | 7 |

## 2. Piece 1: what a stump predicts

**A stump with threshold $t$ and sign $s$ predicts $s$ for every point to the right of $t$ ($x > t$), and $-s$ for the rest.** The sign lets the same threshold point either way.

| $x$ | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| $t = 2.5,\ s = -1$ | +1 | +1 | −1 | −1 | −1 | −1 | −1 | −1 |
| $t = 2.5,\ s = +1$ | −1 | −1 | +1 | +1 | +1 | +1 | +1 | +1 |
| true $y$ | +1 | +1 | −1 | −1 | +1 | +1 | −1 | −1 |

One `np.where` does the whole row: `np.where(x > t, s, -s)` puts `s` where the condition is True and `-s` where it's False.

✏️ **Exercise 1 · Stump predictions.** Return the stump's prediction ($+1$ or $-1$) for every entry of `x`. **Shape: same as `x`.**

**Example**: what goes in, and what should come out:
```python
x = np.arange(1, 9.0)              # [1. 2. 3. 4. 5. 6. 7. 8.]   float, shape (8,)
stump_predict(x, 2.5, -1)          # → [ 1  1 -1 -1 -1 -1 -1 -1]
stump_predict(x, 2.5, 1)           # → [-1 -1  1  1  1  1  1  1]
```

In [ ]:
def stump_predict(x, t, s):
    pass  # your code

In [ ]:
check(1)

## 3. Piece 2: the weighted error

**The weighted error $\varepsilon$ adds up the weights of the points the stump gets wrong.** With equal weights it's just the fraction wrong; after reweighting, a heavy point that's wrong costs more.

$$
\varepsilon = \sum_{i:\ h(x_i) \ne y_i} w_i
$$

Round 1, stump $t = 2.5$, $s = -1$, every weight $\frac{1}{8}$: it gets $x = 5$ and $x = 6$ wrong, so $\varepsilon = \frac{1}{8} + \frac{1}{8} = 0.25$.

```text
mask = (h != y)              [F F F F T T F F]     True where the stump is wrong
w[mask]                      [0.125 0.125]         the weights of the mistakes
w[mask].sum()                0.25                  ε
```

✏️ **Exercise 2 · Weighted error.** `y`: true labels, `h`: a stump's predictions, `w`: weights (all `(n,)`, `w` sums to 1). Return the total weight of the positions where `h != y`. **Shape `()`.**

**Example**: what goes in, and what should come out:
```python
y = np.array([1, 1, -1, -1, 1, 1, -1, -1.0])
h = np.array([1, 1, -1, -1, -1, -1, -1, -1])     # the stump t = 2.5, s = -1
w = np.full(8, 1 / 8)                             # [0.125 0.125 ... 0.125]
weighted_error(y, h, w)                           # → 0.25   (positions 4 and 5 are wrong)
```

In [ ]:
def weighted_error(y, h, w):
    pass  # your code

In [ ]:
check(2)

## 4. Piece 3: searching for the best stump

**Try every midpoint threshold with both signs, and keep the stump with the smallest weighted error.** Eight points give 7 midpoints × 2 signs = 14 candidates.

🖼 **Picture.** Round 1: the weighted error of all 14 stumps. Two stumps tie at 0.25 ($t = 2.5$ and $t = 6.5$, both $s = -1$). The rule "keep the first one found" picks 2.5.

In [ ]:
fig_ada_errors()

Notice the mirror image: flipping the sign turns an error $\varepsilon$ into $1 - \varepsilon$. That's why a stump that's wrong 75% of the time is as useful as one that's right 75% of the time, once you flip it.

✏️ **Exercise 3 · Best stump.** Loop over the midpoints of the sorted unique `x` values and over `s` in `(1, -1)`, in that order. Use your `stump_predict` and `weighted_error`. Return `(t, s, err)` with the smallest error; update only when strictly smaller, so the first one found wins ties.

**Example**: what goes in, and what should come out:
```python
x = np.arange(1, 9.0); y = np.array([1, 1, -1, -1, 1, 1, -1, -1.0])
best_stump(x, y, np.full(8, 1 / 8))       # → (2.5, -1, 0.25)

w2 = np.array([1, 1, 1, 1, 3, 3, 1, 1]) / 12      # round 2's weights: x = 5, 6 count 3 times as much
best_stump(x, y, w2)                       # → (6.5, -1, 0.1667)
```

In [ ]:
def best_stump(x, y, w):
    pass  # your code

In [ ]:
check(3)

## 5. Piece 4: the stump's vote α

**A stump's vote is $\alpha = \frac{1}{2}\ln\frac{1 - \varepsilon}{\varepsilon}$: large when its weighted error is small, 0 for a coin flip, negative when it's worse than guessing.**

🖼 **Picture.** The vote for every possible error. Round 1: $\varepsilon = 0.25$ gives $\alpha = \frac{1}{2}\ln 3 = 0.549$.

In [ ]:
fig_alpha_curve()

**Where it comes from.** AdaBoost picks $\alpha$ to make the **exponential loss** $\sum_i w_i e^{-\alpha y_i h(x_i)}$ as small as possible. Right points have $y_i h(x_i) = +1$ and wrong ones $-1$, so the loss is

$$
\underbrace{(1 - \varepsilon)}_{\text{right weight}}\, e^{-\alpha} + \underbrace{\varepsilon}_{\text{wrong weight}}\, e^{\alpha}
$$

Set the derivative with respect to $\alpha$ to 0: $-(1 - \varepsilon) e^{-\alpha} + \varepsilon e^{\alpha} = 0$, so $e^{2\alpha} = \frac{1 - \varepsilon}{\varepsilon}$, so $\alpha = \frac{1}{2}\ln\frac{1 - \varepsilon}{\varepsilon}$.

✏️ **Exercise 4 · The vote.** Return $\frac{1}{2}\ln\frac{1 - \varepsilon}{\varepsilon}$ for a number `err` between 0 and 1.

**Example**: what goes in, and what should come out:
```python
stump_alpha(0.25)     # → 0.5493   (= 0.5 * ln 3)
stump_alpha(0.5)      # → 0.0      a coin flip gets no vote
stump_alpha(1 / 6)    # → 0.8047
```

In [ ]:
def stump_alpha(err):
    pass  # your code

In [ ]:
check(4)

## 6. Piece 5: reweighting

**Multiply each weight by $e^{-\alpha}$ if the stump got that point right and by $e^{+\alpha}$ if it got it wrong, then divide by the total so the weights add up to 1 again.** One formula does both cases, because $y_i h(x_i)$ is $+1$ for right and $-1$ for wrong:

$$
w_i \leftarrow \frac{w_i\, e^{-\alpha\, y_i h(x_i)}}{\sum_j w_j\, e^{-\alpha\, y_j h(x_j)}}
$$

🖼 **Picture.** Round 1 in three steps: equal weights → multiplied ($\times 0.577$ right, $\times 1.732$ wrong) → divided by their sum 0.866.

In [ ]:
fig_reweight()

**After reweighting, the mistakes always hold exactly half the weight.** Before dividing, the wrong points total $\varepsilon e^{\alpha} = \varepsilon\sqrt{\frac{1 - \varepsilon}{\varepsilon}} = \sqrt{\varepsilon(1 - \varepsilon)}$ and the right ones total $(1 - \varepsilon)e^{-\alpha} = \sqrt{\varepsilon(1 - \varepsilon)}$: the same. So the same stump would now score $\varepsilon = 0.5$, a coin flip, which forces the next round to find a different stump.

✏️ **Exercise 5 · Update the weights.** `w`, `y`, `h` have shape `(n,)`; `alpha` is a number. Return the new weights, normalized to sum to 1. **Shape `(n,)`.** Two lines.

**Example**: what goes in, and what should come out:
```python
w = np.full(8, 1 / 8)
y = np.array([1, 1, -1, -1, 1, 1, -1, -1.0])
h = np.array([1, 1, -1, -1, -1, -1, -1, -1])
update_weights(w, y, h, 0.5493)
# → [0.0833 0.0833 0.0833 0.0833 0.25 0.25 0.0833 0.0833]    right → 1/12, wrong → 1/4
```

In [ ]:
def update_weights(w, y, h, alpha):
    pass  # your code

In [ ]:
check(5)

## 7. Putting it together, and predicting

With the four pieces written, `adaboost_fit` is the five-line loop from section 1. One safety detail: if a stump is perfect, $\varepsilon = 0$ and the log divides by zero, so clip $\varepsilon$ into $[10^{-10},\ 1 - 10^{-10}]$ first.

| Round | Weights in | Stump | $\varepsilon$ | $\alpha$ | Mistakes |
|---|---|---|---|---|---|
| 1 | all 1/8 | t = 2.5, s = −1 | 0.250 | 0.549 | x = 5, 6 |
| 2 | 1/12, … , 1/4 at x = 5, 6 | t = 6.5, s = −1 | 0.167 | 0.805 | x = 3, 4 |
| 3 | 0.05, 0.05, 0.25, 0.25, 0.15, 0.15, 0.05, 0.05 | t = 4.5, s = +1 | 0.200 | 0.693 | x = 1, 2, 7, 8 |

✏️ **Exercise 6 · AdaBoost fit.** Start with equal weights. Each round: `best_stump`, clip the error, `stump_alpha`, `update_weights` (with that stump's predictions), and append `(t, s, alpha)`. Return the list.

**Example**: what goes in, and what should come out:
```python
adaboost_fit(np.arange(1, 9.0), np.array([1, 1, -1, -1, 1, 1, -1, -1.0]), 3)
# → [(2.5, -1, 0.5493), (6.5, -1, 0.8047), (4.5, 1, 0.6931)]     a list of 3 tuples
```

In [ ]:
def adaboost_fit(x, y, rounds):
    pass  # your code

In [ ]:
check(6)

**Predicting:** add up every stump's prediction times its vote, $F(x) = \sum_t \alpha_t h_t(x)$, and take the sign (a score of exactly 0 counts as $+1$).

✏️ **Exercise 7 · AdaBoost predict.** `model` is the list from exercise 6. Return $+1$ where $F(x) \ge 0$ and $-1$ elsewhere. **Shape: same as `x`.**

**Example**: what goes in, and what should come out:
```python
model = [(2.5, -1, 0.5493), (6.5, -1, 0.8047), (4.5, 1, 0.6931)]
adaboost_predict(model, np.arange(1, 9.0))      # → [ 1  1 -1 -1  1  1 -1 -1]   all 8 right
# x = 3: F = -0.549 + 0.805 - 0.693 = -0.438 → -1
```

In [ ]:
def adaboost_predict(model, x):
    pass  # your code

In [ ]:
check(7)

## 8. Gradient boosting: the whole loop first

> ↩ **Recap** · [Week 3 · 06 §6 (gradient boosting by hand) ↗](../week3/06_ensembles.ipynb)
>
> $x = (1, 2, 3, 4)$, $y = (1, 3, 6, 10)$, stumps, $\eta = 0.5$: $F_0 = 5$, then $F_1 = (3.5, 3.5, 6.5, 6.5)$, then $F_2 = (2.917, 2.917, 5.917, 8.25)$. MSE $11.5 \to 4.75 \to 1.69$.

**Gradient boosting starts from one constant prediction and repeatedly adds a small stump that predicts what's still wrong.**

🖼 **Picture.** Top: the prediction $F$ (orange steps) after 0, 1 and 2 rounds; red dotted lines are the residuals. Bottom: each round's stump (blue) is fitted to the residual bars above it.

In [ ]:
fig_gb_steps()

**Why start from the mean?** The constant $c$ with the smallest squared error solves $\frac{d}{dc}\sum_i (y_i - c)^2 = -2\sum_i (y_i - c) = 0$, i.e. $c = \bar y$. Here $\bar y = \frac{1 + 3 + 6 + 10}{4} = 5$.

| Line | In words | Section |
|---|---|---|
| `F = np.full(n, y.mean())` | start from the mean | (given) |
| `r = residuals(y, F)` | what's still wrong | 9 |
| `t, a, b = fit_reg_stump(x, r)` | the stump that best predicts `r` | 10 |
| `F = F + lr * reg_stump_predict(x, t, a, b)` | move a fraction of the way | 11 |

## 9. Piece 1: residuals

**A residual is the true value minus the current prediction: positive means "predict higher", negative means "predict lower".** For the squared loss it's also the negative gradient ([Week 3 · 06 §6 ↗](../week3/06_ensembles.ipynb)), which is where the name *gradient* boosting comes from.

✏️ **Exercise 8 · Residuals.** Return `y - F`. **Shape `(n,)`.** One line, but give it a name: the loop reads better.

**Example**: what goes in, and what should come out:
```python
residuals(np.array([1, 3, 6, 10.0]), np.full(4, 5.0))     # → [-4. -2.  1.  5.]
```

In [ ]:
def residuals(y, F):
    pass  # your code

In [ ]:
check(8)

## 10. Piece 2: a stump that predicts the residuals

**A regression stump splits at the threshold $t$ with the smallest total SSE, and predicts the mean residual on each side.** It's the regression tree from [Week 3 · 04 §5 ↗](../week3/04_decision_trees.ipynb) stopped after one split.

| Threshold | Left residuals | Left mean | Right residuals | Right mean | Total SSE |
|---|---|---|---|---|---|
| 1.5 | −4 | −4.00 | −2, 1, 5 | 1.33 | 0 + 24.67 = 24.67 |
| **2.5** | −4, −2 | **−3.00** | 1, 5 | **3.00** | 2 + 8 = **10.00** |
| 3.5 | −4, −2, 1 | −1.67 | 5 | 5.00 | 12.67 + 0 = 12.67 |

✏️ **Exercise 9 · Fit a regression stump.** `x` and `r` have shape `(n,)`. Try every midpoint of the sorted unique `x`; for each, compute the left and right means of `r` and the total SSE. Return `(t, left_mean, right_mean)` for the smallest SSE (first one wins ties).

**Example**: what goes in, and what should come out:
```python
fit_reg_stump(np.array([1, 2, 3, 4.0]), np.array([-4, -2, 1, 5.0]))
# → (2.5, -3.0, 3.0)
fit_reg_stump(np.array([1, 2, 3, 4.0]), np.array([-2.5, -0.5, -0.5, 3.5]))
# → (3.5, -1.1667, 3.5)     round 2's stump
```

In [ ]:
def fit_reg_stump(x, r):
    pass  # your code

In [ ]:
check(9)

✏️ **Exercise 10 · Use a regression stump.** Return `left_value` where `x <= t` and `right_value` elsewhere. **Shape: same as `x`.**

**Example**: what goes in, and what should come out:
```python
reg_stump_predict(np.array([1, 2, 3, 4.0]), 2.5, -3.0, 3.0)    # → [-3. -3.  3.  3.]
```

In [ ]:
def reg_stump_predict(x, t, left_value, right_value):
    pass  # your code

In [ ]:
check(10)

## 11. Piece 3: the update, and predicting

**Add the stump's prediction, scaled down by the learning rate $\eta$, to every point's prediction.** With $\eta = 0.5$, round 1 adds $0.5 \times (-3, -3, 3, 3)$ to $(5, 5, 5, 5)$, giving $(3.5, 3.5, 6.5, 6.5)$.

✏️ **Exercise 11 · Gradient boosting fit.** Start from `f0 = y.mean()` and `F` = that value for every point. Each round: residuals → `fit_reg_stump` → update `F`. Store each stump as the tuple `(t, a, b)`. Return `(f0, stumps)`.

**Example**: what goes in, and what should come out:
```python
f0, stumps = gb_fit(np.array([1, 2, 3, 4.0]), np.array([1, 3, 6, 10.0]), 2, 0.5)
f0        # → 5.0
stumps    # → [(2.5, -3.0, 3.0), (3.5, -1.1667, 3.5)]
```

In [ ]:
def gb_fit(x, y, rounds, lr):
    pass  # your code

In [ ]:
check(11)

✏️ **Exercise 12 · Gradient boosting predict.** $F(x) = f_0 + \eta \sum_m h_m(x)$, using `reg_stump_predict` for each stored stump. **Shape: same as `x`.**

**Example**: what goes in, and what should come out:
```python
gb_predict(5.0, [(2.5, -3.0, 3.0), (3.5, -1.1667, 3.5)], 0.5, np.array([1, 2, 3, 4.0]))
# → [2.9167 2.9167 5.9167 8.25  ]
```

In [ ]:
def gb_predict(f0, stumps, lr, x):
    pass  # your code

In [ ]:
check(12)

In [ ]:
for n in range(1, 13):
    check(n)

## 12. Side by side

|  | AdaBoost | Gradient boosting |
|---|---|---|
| task | classification, $y \in \{-1, +1\}$ | regression here (any loss, in general) |
| what changes each round | the **weights** of the points | the **targets**: the residuals |
| each stump is fitted to | the labels, with weights | the residuals, without weights |
| how much a stump counts | its vote $\alpha$, from its error | the learning rate $\eta$, the same every round |
| final model | sign of $\sum_t \alpha_t h_t(x)$ | $f_0 + \eta \sum_m h_m(x)$ |
| loss it minimizes | exponential, $e^{-y F(x)}$ | squared error here; any differentiable loss in general |

**They're the same idea.** Both build $F$ as a sum of small trees, each one chosen to reduce a loss given the trees so far. AdaBoost turns out to be gradient boosting with the exponential loss, written in its own notation. XGBoost and LightGBM are gradient boosting with faster split search and better steps ([Week 3 · 06 §6 ↗](../week3/06_ensembles.ipynb)).

---
Solutions: `solutions/01_solutions.py`. Next: `02_pca.ipynb`.